## START

In [1]:
# We want everyone to start with the same code, so we prepared a starter package.

# Download it:

# PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring'
# !wget $PREFIX/rag_helper.py
# !wget $PREFIX/starter.py


# Like previously, you can use any alternative you want.

# The starter loads the 72 course lessons, builds a text-search index, and wraps it in a RAGBase instance you can call right away:

from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by checking whether the model returned any `function_call` items.

- If it **did** return a function call, the code runs the tool, appends the tool result to `messages`, and continues the loop.
- If it **did not** return any function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps looping until the model replies with a final message and no more tool calls.


In [2]:
# OpenTelemetry setup
# First, install the OpenTelemetry libraries:

# uv add opentelemetry-api opentelemetry-sdk

# -- opentelemetry-api is the interface - the classes and functions you import in your code (trace, Tracer, Span)
# -- opentelemetry-sdk is the implementation that actually creates and processes spans.

# OpenTelemetry
# Before we start, we need to learn a few concepts from OTel - we will use them in this homework.

# A trace is the end-to-end story of a single request as it moves through your system. For us, it's one RAG call.
# A span is one operation within a trace. A trace is made of one or more spans, organized as a tree. Each span has a name, 
# a start and end time, and a set of attributes. 
# For us we will have one span inside the trace, but for agents one trace will have multiple spans.

# Attributes are key-value pairs attached to a span - anything you want to record, like the number of tokens used or the cost of a call.
# When a span finishes - meaning the code block it wraps completes - the SDK hands it to a span processor, which forwards it to an exporter. 
# The exporter decides where the span goes: to the console, to a file, to a database, or to a remote collector. 
# We will see all of this in practice in the questions below.

# We start with the ConsoleSpanExporter, which prints each finished span to the terminal so we can see what OTel captures:

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
tracer

In [4]:
# Here is what each line does:

# TracerProvider() creates the SDK's central configuration object. It owns the span processors and decides how spans are built.
# SimpleSpanProcessor(ConsoleSpanExporter()) wires a processor that forwards every finished span to the console exporter, one at a time. 
# "Simple" means synchronous and immediate - good for development.

# trace.set_tracer_provider(provider) registers the provider globally, so every call to trace.get_tracer(...) returns a tracer backed by it.

# trace.get_tracer("llm-zoomcamp") returns a Tracer we use to create spans. 
# The string is just a label for the instrumentation scope - it identifies which part of the code produced the spans.
# Put this block at the top of your script, before you import or use starter - 
# so the tracer provider is ready before any code that might create spans.

# With the tracer in hand, you can wrap any block of code in a span:
with tracer.start_as_current_span("q1_rag") as span:
    # your code here --
    answer = rag.rag(query)
    print(answer)
    print("****************************************************")

    # span.set_attribute("my_key", "my_value")
    # span.set_attribute("input_tokens", usage.input_tokens)
    # span.set_attribute("output_tokens", usage.output_tokens)


The loop keeps calling the model by checking whether the response contains any `function_call` items.

- It sends the current `messages` to the model.
- If the model returns a `function_call`, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- After processing the response, it does:

```python
if has_function_calls == False:
    break
```

So it continues looping as long as the model keeps asking for tools, and stops when the model returns a message with no function calls.
****************************************************
{
    "name": "q1_rag",
    "context": {
        "trace_id": "0x97a104e7acbf7c4bf4b09b0dc4bf709a",
        "span_id": "0xc5f769f17db606b3",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-21T19:06:49.891185Z",
    "end_time": "2026-07-21T19:06:51.983018Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": 

## Q1

In [5]:
# Q1. First trace
# Wrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of 
# RAGBase that wraps rag(), search(), and llm() each in their own span.

# Run this query:
# How does the agentic loop keep calling the model until it stops?

# The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. 
# How many spans does the trace produce?

# 1
# 3 -- in fact 4 ))
# 5
# 7


from rag_helper import RAGBase


class RAGTraced(RAGBase):
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)

            results = super().search(query, num_results)

            span.set_attribute("results.count", len(results))
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("model", self.model)

            response = super().llm(prompt)

            # Capture token usage
            if response.usage:
                usage = response.usage

                span.set_attribute(
                    "input_tokens",
                    usage.input_tokens
                )
                span.set_attribute(
                    "output_tokens",
                    usage.output_tokens
                )
                span.set_attribute(
                    "total_tokens",
                    usage.total_tokens
                )

            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)

            response = super().rag(query)

            span.set_attribute("response.length", len(response))
            return response



In [6]:
from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

from dotenv import load_dotenv
load_dotenv()

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()

rag = RAGTraced(index=index, llm_client=client)

with tracer.start_as_current_span("q1_rag") as span:
    # your code here --
    answer = rag.rag(query)
    print(answer)
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

    

{
    "name": "search",
    "context": {
        "trace_id": "0xa44d00bb52d9d6a2f8ffeae4f836380e",
        "span_id": "0xcd811ed8f7fa6e09",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x9ac4c28f4fdf03f2",
    "start_time": "2026-07-21T19:06:52.791947Z",
    "end_time": "2026-07-21T19:06:52.793181Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "results.count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "6ab0a124-af8e-4b04-b7e5-c1d05f6e4287",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "tra

## Q2

In [8]:
# Q2. Capturing metrics as span attributes
# Spans are not just timing markers - you can attach any information you want to them with set_attribute. 
# We already use spans to record how long each step takes. Now we'll add the metrics we care about: tokens and cost.

# Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) 
# and set them as attributes on the llm span:

# span.set_attribute("input_tokens", usage.input_tokens)
# span.set_attribute("output_tokens", usage.output_tokens)
# And since we know both input and output tokens, we can also compute the cost using the code from the previous modules.

# Now re-run the query. How many input tokens do we see?

# 700
# 7000 -- 7111
# 70000
# 700000
# These numbers vary between runs. Pick the closest option.

with tracer.start_as_current_span("q1_rag") as span:
    # your code here --
    answer = rag.rag(query)
    print(answer)
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++")


  # "attributes": {
  #       "model": "gpt-5.4-mini",
  #       "input_tokens": 7111,
  #       "output_tokens": 123,
  #       "total_tokens": 7234
  #   },

{
    "name": "search",
    "context": {
        "trace_id": "0xf575e816c3bd61fa711ba68a4b58c8ac",
        "span_id": "0x3fcde882640cf2d5",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xbfd639a2e09bfbec",
    "start_time": "2026-07-21T19:11:02.074348Z",
    "end_time": "2026-07-21T19:11:02.076238Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "results.count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "6ab0a124-af8e-4b04-b7e5-c1d05f6e4287",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "tra

## Q3

In [9]:
# Q3. Span timing
# Each span automatically records its duration. Look at the console output from Q1 and find the durations for the search span and the llm span.

# For a typical query, roughly how long does the LLM call take?

# Under 100ms
# 100-500ms
# 500-2000ms - about  1 sec in the second call, 2 sec first ...
# Over 2000ms
# The first call can be slower (cold start). Pick the range you see most often.

    #  "start_time": "2026-07-21T19:11:02.077163Z",
    # "end_time": "2026-07-21T19:11:03.769614Z"

## Q4 - SQL lite